In [39]:
#%conda install PyVCF

Solving environment: done


==> WARNING: A newer version of conda exists. <==
  current version: 22.9.0
  latest version: 24.3.0

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.

Retrieving notices: ...working... done

Note: you may need to restart the kernel to use updated packages.


In [40]:
import io
import os
import pandas as pd
import re
import vcf

In [92]:
def read_vcf(path):
    with open(path, 'r') as f:
        lines = [l for l in f if not l.startswith('##')]
    return pd.read_csv(
        io.StringIO(''.join(lines)),
        dtype={'#CHROM': str, 'POS': int, 'ID': str, 'REF': str, 'ALT': str,
               'QUAL': str, 'FILTER': str, 'INFO': str},
        sep='\t'
    ).rename(columns={'#CHROM': 'CHROM'})

def extract_af(info):
    match = re.search(r'AF=([\d\.]+)', info)
    return match.group(1) if match else None

def filter_vcf(vcf_file):
    vcf_reader = vcf.Reader(open(vcf_file, 'r'))

    for record in vcf_reader:
        if 'AF' in record.INFO and record.INFO['AF'] > 0.5:
            print(record)

def fasta_to_all_chars_dataframe(filepath):
    """ 
    Parses a FASTA file and returns a DataFrame with all characters enumerated
    """
    with open(filepath, 'r') as file:
        position = 1  
        data = []  
        for line in file:
            for char in line:
                if char == '\n':
                    data.append((position, 'line_break', '\\n'))
                else:
                    data.append((position, 'sequence' if char != '>' else 'header', char))
                position += 1

    return pd.DataFrame(data, columns=['POS', 'Type', 'REF'])

In [56]:
df = read_vcf('/Users/fionachow/Documents/NYU/CDS/Spring 2024/Research Fair/Hochwagen/Individual Data/ERR3240115/ERR3240115_rDNA.vcf') #vcf file from original variant calling of 1 individual

df.head(32)

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
0,Human,31,.,T,C,21247,PASS,"DP=1510;AF=1.000000;SB=0;DP4=0,0,1509,1"
1,Human,32,.,C,T,22076,PASS,"DP=1554;AF=1.000000;SB=0;DP4=0,0,1553,1"
2,Human,73,.,C,A,46496,PASS,"DP=2625;AF=0.996952;SB=0;DP4=7,0,2607,10"
3,Human,74,.,A,C,48045,PASS,"DP=2657;AF=0.999624;SB=0;DP4=1,0,2646,10"
4,Human,80,.,A,AC,49314,PASS,"DP=2829;AF=0.977024;SB=0;DP4=64,0,2747,17;INDE..."
5,Human,103,.,GC,G,49314,PASS,"DP=3376;AF=0.987559;SB=31;DP4=38,4,3304,30;IND..."
6,Human,121,.,C,G,49314,PASS,"DP=3853;AF=0.998962;SB=0;DP4=0,0,3779,70"
7,Human,209,.,C,G,2469,PASS,"DP=4542;AF=0.109203;SB=2;DP4=3584,459,436,60"
8,Human,237,.,G,GC,49314,PASS,"DP=4988;AF=0.966921;SB=14;DP4=164,21,3988,835;..."
9,Human,270,.,C,CG,49314,PASS,"DP=5712;AF=0.945553;SB=6;DP4=242,88,4122,1279;..."


In [50]:
df[df['REF'].str.len() > 1] #checks for reference alleles having more than 1 base. Why does lofreq report 'REF' this way?

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
5,Human,103,.,GC,G,49314,PASS,"DP=3376;AF=0.987559;SB=31;DP4=38,4,3304,30;IND..."
19,Human,537,.,TG,T,49314,PASS,"DP=9753;AF=0.957346;SB=86;DP4=310,144,5049,428..."
24,Human,633,.,GT,G,529,PASS,"DP=9666;AF=0.052245;SB=10;DP4=4846,4397,284,22..."
28,Human,763,.,CGG,C,49314,PASS,"DP=9684;AF=0.649318;SB=75;DP4=1732,1687,2816,3..."
31,Human,963,.,TGAGAC,T,394,PASS,"DP=9244;AF=0.008979;SB=14;DP4=4474,4750,50,33;..."
...,...,...,...,...,...,...,...,...
318,Human,11069,.,GC,G,28436,PASS,"DP=2541;AF=0.935852;SB=2;DP4=66,133,838,1540;I..."
323,Human,11118,.,GC,G,17778,PASS,"DP=2197;AF=0.863450;SB=0;DP4=161,173,931,966;I..."
329,Human,11268,.,CG,C,19335,PASS,"DP=3057;AF=0.718024;SB=1045;DP4=790,104,1061,1..."
333,Human,11417,.,GGCA,G,3864,PASS,"DP=3769;AF=0.205890;SB=0;DP4=1936,1082,496,280..."


In [69]:
df[(df['REF'].str.len() == 4) & (df['ALT'].str.len() == 2)]


,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO


In [43]:
#extracting AF into a seperate column
df['AF'] = df['INFO'].apply(extract_af)

df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,AF
0,Human,31,.,T,C,21247,PASS,"DP=1510;AF=1.000000;SB=0;DP4=0,0,1509,1",1.000000
1,Human,32,.,C,T,22076,PASS,"DP=1554;AF=1.000000;SB=0;DP4=0,0,1553,1",1.000000
2,Human,73,.,C,A,46496,PASS,"DP=2625;AF=0.996952;SB=0;DP4=7,0,2607,10",0.996952
3,Human,74,.,A,C,48045,PASS,"DP=2657;AF=0.999624;SB=0;DP4=1,0,2646,10",0.999624
4,Human,80,.,A,AC,49314,PASS,"DP=2829;AF=0.977024;SB=0;DP4=64,0,2747,17;INDE...",0.977024
...,...,...,...,...,...,...,...,...,...
374,Human,13219,.,C,CG,43670,PASS,"DP=2182;AF=0.944088;SB=1;DP4=1,152,29,2031;IND...",0.944088
375,Human,13235,.,GC,G,35659,PASS,"DP=1803;AF=0.953411;SB=22;DP4=5,91,18,1701;IND...",0.953411
376,Human,13278,.,G,GGC,13133,PASS,"DP=1033;AF=0.909971;SB=0;DP4=0,103,2,938;INDEL...",0.909971
377,Human,13280,.,G,C,13572,PASS,"DP=1017;AF=1.000000;SB=0;DP4=0,0,2,1015",1.000000


In [44]:
type(df.loc[2, 'AF']) #string type

str

In [45]:
df['AF'] = df['AF'].astype(float) #converting format to float

type(df.loc[2, 'AF'])

numpy.float64

In [46]:
filtered_df = df[df['AF'] > 0.5] #filtering for AF threshold >0.5; extracting majority alleles at each position from variant calling

filtered_df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,AF
0,Human,31,.,T,C,21247,PASS,"DP=1510;AF=1.000000;SB=0;DP4=0,0,1509,1",1.000000
1,Human,32,.,C,T,22076,PASS,"DP=1554;AF=1.000000;SB=0;DP4=0,0,1553,1",1.000000
2,Human,73,.,C,A,46496,PASS,"DP=2625;AF=0.996952;SB=0;DP4=7,0,2607,10",0.996952
3,Human,74,.,A,C,48045,PASS,"DP=2657;AF=0.999624;SB=0;DP4=1,0,2646,10",0.999624
4,Human,80,.,A,AC,49314,PASS,"DP=2829;AF=0.977024;SB=0;DP4=64,0,2747,17;INDE...",0.977024
...,...,...,...,...,...,...,...,...,...
374,Human,13219,.,C,CG,43670,PASS,"DP=2182;AF=0.944088;SB=1;DP4=1,152,29,2031;IND...",0.944088
375,Human,13235,.,GC,G,35659,PASS,"DP=1803;AF=0.953411;SB=22;DP4=5,91,18,1701;IND...",0.953411
376,Human,13278,.,G,GGC,13133,PASS,"DP=1033;AF=0.909971;SB=0;DP4=0,103,2,938;INDEL...",0.909971
377,Human,13280,.,G,C,13572,PASS,"DP=1017;AF=1.000000;SB=0;DP4=0,0,2,1015",1.000000


In [51]:
#alt way to read vcf files filtered for variant positions
filter_vcf('/Users/fionachow/Documents/NYU/CDS/Spring 2024/Research Fair/Hochwagen/Individual Data/ERR3240115/ERR3240115_rDNA.vcf') #similar outputs to the pandas filter

Record(CHROM=Human, POS=31, REF=T, ALT=[C])
Record(CHROM=Human, POS=32, REF=C, ALT=[T])
Record(CHROM=Human, POS=73, REF=C, ALT=[A])
Record(CHROM=Human, POS=74, REF=A, ALT=[C])
Record(CHROM=Human, POS=80, REF=A, ALT=[AC])
Record(CHROM=Human, POS=103, REF=GC, ALT=[G])
Record(CHROM=Human, POS=121, REF=C, ALT=[G])
Record(CHROM=Human, POS=237, REF=G, ALT=[GC])
Record(CHROM=Human, POS=270, REF=C, ALT=[CG])
Record(CHROM=Human, POS=275, REF=C, ALT=[CG])
Record(CHROM=Human, POS=284, REF=A, ALT=[AC])
Record(CHROM=Human, POS=363, REF=G, ALT=[GT])
Record(CHROM=Human, POS=386, REF=T, ALT=[TGGCC])
Record(CHROM=Human, POS=419, REF=C, ALT=[CGT])
Record(CHROM=Human, POS=517, REF=G, ALT=[GC])
Record(CHROM=Human, POS=522, REF=C, ALT=[CG])
Record(CHROM=Human, POS=524, REF=C, ALT=[G])
Record(CHROM=Human, POS=537, REF=TG, ALT=[T])
Record(CHROM=Human, POS=560, REF=C, ALT=[G])
Record(CHROM=Human, POS=561, REF=G, ALT=[GT])
Record(CHROM=Human, POS=623, REF=C, ALT=[CT])
Record(CHROM=Human, POS=626, REF=T, ALT=[G

In [ ]:
mpileup_file_path = '/Users/fionachow/Documents/NYU/CDS/Spring 2024/Research Fair/Hochwagen/Individual Data/ERR3240115/ERR3240115_rDNA.mpileup' #extracting reference alleles for all Positions

mpileup_df = pd.read_csv(mpileup_file_path, sep='\t', header=None, usecols=[0, 1, 2], names=['CHROM', 'POS', 'REF'])

mpileup_df.head(32)

,CHROM,POS,REF
0,Human,1,G
1,Human,2,C
2,Human,3,T
3,Human,4,G
4,Human,5,A
5,Human,6,C
6,Human,7,A
7,Human,8,C
8,Human,9,G
9,Human,10,C


In [75]:
mpileup_df.shape

(13314, 3)

In [55]:
mpileup_df[mpileup_df['REF'].str.len() > 1] #checking each reference position has 1 base

,CHROM,POS,REF


In [70]:
merged_df = mpileup_df.merge(filtered_df[['POS', 'ALT']], on='POS', how='left') #merge with filtered_df i.e. alt(majority) alleles at each position from variant calling

merged_df.head(563)

,CHROM,POS,REF,ALT
0,Human,1,G,NaN
1,Human,2,C,NaN
2,Human,3,T,NaN
3,Human,4,G,NaN
4,Human,5,A,NaN
...,...,...,...,...
558,Human,559,C,NaN
559,Human,560,C,G
560,Human,561,G,GT
561,Human,562,G,NaN


In [71]:
merged_df['REF'] = merged_df.apply(lambda row: row['ALT'] if pd.notnull(row['ALT']) else row['REF'], axis=1) #replace ref values w alt values if not NaN

merged_df.head(563)

,CHROM,POS,REF,ALT
0,Human,1,G,NaN
1,Human,2,C,NaN
2,Human,3,T,NaN
3,Human,4,G,NaN
4,Human,5,A,NaN
...,...,...,...,...
558,Human,559,C,NaN
559,Human,560,G,G
560,Human,561,GT,GT
561,Human,562,G,NaN


In [ ]:
merged_df.drop('ALT', axis=1, inplace=True)

merged_df

,CHROM,POS,REF
0,Human,1,G
1,Human,2,C
2,Human,3,T
3,Human,4,G
4,Human,5,A
...,...,...,...
13309,Human,13310,T
13310,Human,13311,C
13311,Human,13312,G
13312,Human,13313,C


In [ ]:
# merged_df.to_csv('reference.csv', index=False)


In [88]:
#alt way to get reference enumerated
df_fasta = fasta_to_all_chars_dataframe('/Users/fionachow/Documents/rDNA_prototype_prerRNA_only.fa') #manually deleted first line of header

print(df_fasta.head())
print(df_fasta.shape) 

   POS      Type REF
0    1  sequence   G
1    2  sequence   C
2    3  sequence   T
3    4  sequence   G
4    5  sequence   A
(13314, 3)


In [90]:
assert mpileup_df['POS'].equals(df_fasta['POS']) and mpileup_df['REF'].equals(df_fasta['REF']) #checks my workaround matches enumerating reference file
